# Perturbation Sensitivity — Do Sinks Limit Information Spread? (GRIT)

Replicating Barbero et al. Figure 2 for graphs using GRIT (no MPNN).
Since GRIT has no local message passing, perturbation can only spread through
global attention — a cleaner test than GPS.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import yaml
from tqdm import tqdm

from src.grit_model import InstrumentedGRIT
from src.datasets import get_dataloaders, DATASET_INFO

matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 150,
                             'axes.spines.top': False, 'axes.spines.right': False})

OUTPUTS_DIR = '../outputs'
FIGURES_DIR = '../outputs/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
def load_experiment(experiment_id, device):
    config_path = os.path.join(OUTPUTS_DIR, experiment_id, 'config.yaml')
    with open(config_path) as f:
        config = yaml.safe_load(f)
    dataset_info = DATASET_INFO[config['data']['dataset']].copy()
    vnode_cfg = config.get('vnode', {})
    if vnode_cfg.get('enabled', False) and dataset_info.get('num_node_types') is not None:
        dataset_info['num_node_types'] = dataset_info['num_node_types'] + 1
        dataset_info['num_edge_types'] = dataset_info['num_edge_types'] + 1
    model = InstrumentedGRIT(config, dataset_info).to(device)
    model.load_state_dict(torch.load(
        os.path.join(OUTPUTS_DIR, experiment_id, 'best_model.pt'),
        map_location=device, weights_only=True))
    model.eval()
    return model, config, dataset_info

## 1. Perturbation experiment

In [ ]:
@torch.no_grad()
def run_perturbation_experiment(model, config, device, max_graphs=100, noise_scale=0.1):
    _, _, test_loader, _ = get_dataloaders(config)
    model.eval()
    num_layers = model.num_layers
    spread = {l: {} for l in range(num_layers + 1)}
    graphs_done = 0
    for batch in tqdm(test_loader, desc='Perturbation'):
        if graphs_done >= max_graphs: break
        batch = batch.to(device)
        _ = model(batch, collect_diagnostics=True)
        clean_reps = {l: model.layer_data[l]['h'].clone() for l in range(num_layers + 1)}
        clean_batch_ids = model.layer_data[0]['batch']
        for g_idx, g_id in enumerate(clean_batch_ids.unique()):
            if graphs_done >= max_graphs: break
            graph_mask = (clean_batch_ids == g_id)
            node_indices = torch.where(graph_mask)[0]
            num_nodes_g = len(node_indices)
            if num_nodes_g < 3: continue
            perturb_local = torch.randint(0, num_nodes_g, (1,)).item()
            perturb_global = node_indices[perturb_local].item()
            g_edges = batch.edge_index[:, batch.batch[batch.edge_index[0]] == g_id]
            local_map = {int(n): i for i, n in enumerate(node_indices)}
            adj = {i: set() for i in range(num_nodes_g)}
            for s, d in g_edges.t().tolist():
                if s in local_map and d in local_map:
                    adj[local_map[s]].add(local_map[d])
            distances = [-1] * num_nodes_g
            distances[perturb_local] = 0
            queue = [perturb_local]; head = 0
            while head < len(queue):
                curr = queue[head]; head += 1
                for nbr in adj[curr]:
                    if distances[nbr] == -1:
                        distances[nbr] = distances[curr] + 1
                        queue.append(nbr)
            batch_p = batch.clone()
            if batch_p.x.dtype == torch.long:
                batch_p.x[perturb_global] = torch.randint(0, 28, batch_p.x[perturb_global].shape, device=batch_p.x.device)
            else:
                noise = torch.randn_like(batch_p.x[perturb_global:perturb_global+1].float()) * noise_scale
                batch_p.x[perturb_global] = batch_p.x[perturb_global].float() + noise.squeeze()
            _ = model(batch_p, collect_diagnostics=True)
            perturbed_reps = {l: model.layer_data[l]['h'].clone() for l in range(num_layers + 1)}
            for l in range(num_layers + 1):
                delta = (clean_reps[l][graph_mask] - perturbed_reps[l][graph_mask]).float().norm(dim=-1)
                for ni in range(num_nodes_g):
                    dist = distances[ni]
                    if dist < 0: continue
                    spread[l].setdefault(dist, []).append(delta[ni].item())
            graphs_done += 1
    return spread

In [ ]:
perturbation_results = {}
for eid in ['zinc-grit', 'zinc-grit-vnode', 'peptides-grit', 'peptides-grit-vnode']:
    if not os.path.exists(os.path.join(OUTPUTS_DIR, eid, 'best_model.pt')):
        print(f'Skipping {eid} — not trained yet')
        continue
    print(f'\n=== {eid} ===')
    model, config, _ = load_experiment(eid, device)
    perturbation_results[eid] = run_perturbation_experiment(model, config, device)
    del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 2. Perturbation heatmaps and line plots

In [ ]:
def spread_to_heatmap(spread, max_dist=8):
    num_layers = max(spread.keys()) + 1
    hm = np.zeros((num_layers, max_dist + 1))
    for l in range(num_layers):
        for d in range(max_dist + 1):
            vals = spread[l].get(d, [])
            hm[l, d] = np.mean(vals) if vals else 0.0
    return hm

# Plot heatmaps for each available pair
for ds_prefix, ds_label, max_d in [('zinc-grit', 'ZINC', 6), ('peptides-grit', 'Peptides', 8)]:
    novn = ds_prefix
    vn = ds_prefix + '-vnode'
    avail = [e for e in [novn, vn] if e in perturbation_results]
    if not avail: continue
    
    fig, axes = plt.subplots(1, len(avail), figsize=(7 * len(avail), 5))
    if len(avail) == 1: axes = [axes]
    hms = [spread_to_heatmap(perturbation_results[e], max_d) for e in avail]
    vmax = max(h.max() for h in hms)
    for ax, hm, eid in zip(axes, hms, avail):
        label = 'VNode' if 'vnode' in eid else 'No VNode'
        im = ax.imshow(hm.T, aspect='auto', origin='lower', cmap='YlOrRd', vmin=0, vmax=vmax)
        ax.set_xlabel('Layer'); ax.set_ylabel('Graph distance')
        ax.set_title(f'{ds_label} — {label} (GRIT)')
        ax.set_yticks(range(max_d + 1))
    plt.colorbar(im, ax=axes, label='Mean $\\|\\Delta h\\|$')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f'fig7_perturbation_{ds_prefix.replace("-grit", "")}.pdf'),
                bbox_inches='tight', dpi=150)
    plt.show()

## 3. Containment ratios

In [ ]:
print(f"{'Experiment':<24} {'d=0 (last L)':>14} {'d=3+ (last L)':>14} {'Ratio':>8}")
print('=' * 65)
for eid, spread in perturbation_results.items():
    num_layers = max(spread.keys())
    d0 = np.mean(spread[num_layers].get(0, [0]))
    d3_vals = []
    for d in range(3, 10): d3_vals.extend(spread[num_layers].get(d, []))
    d3 = np.mean(d3_vals) if d3_vals else 0.0
    ratio = d0 / d3 if d3 > 1e-8 else float('inf')
    print(f'{eid:<24} {d0:>14.4f} {d3:>14.4f} {ratio:>8.2f}')
print('\nHigher ratio = more localised (less over-mixing)')